In [1]:
# ==============================================================================
# CHOLEC80 — BACKBONE FINE-TUNING & FEATURE EXTRACTION (STEP 3)
# ==============================================================================
# Esegui questo codice in un NUOVO Notebook Kaggle, senza la TCN e la P100 GPU per evitare l'errore di PyTorch.
# ==============================================================================

import os, cv2, warnings, shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms
from PIL import Image
from sklearn.metrics import f1_score
from tqdm.auto import tqdm
from copy import deepcopy

warnings.filterwarnings('ignore')

In [2]:
# --- 1. CONFIGURAZIONE E SPLIT ---

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

BASE_DIR = "/kaggle/input/datasets/christiancorvino/cholec80-mr-sapienza"
if not os.path.exists(BASE_DIR):
    BASE_DIR = "/kaggle/input/cholec80-mr-sapienza"

PHASE_DIR = os.path.join(BASE_DIR, "phase_annotations")
VIDEO_DIR = os.path.join(BASE_DIR, "videos")

OUTPUT_DIR = "/kaggle/working/features_finetuned"
CACHE_DIR = "/kaggle/working/frame_cache"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

PHASE_DICT = {
    'Preparation': 0, 'CalotTriangleDissection': 1, 'ClippingCutting': 2,
    'GallbladderDissection': 3, 'GallbladderPackaging': 4,
    'CleaningCoagulation': 5, 'GallbladderRetraction': 6
}
NUM_CLASSES = len(PHASE_DICT)

# SPLIT RIGIDO: 1-27 (CNN Train), 28-32 (CNN Val interna)
CNN_TRAIN_IDS = list(range(1, 28))
CNN_VAL_IDS = list(range(28, 33))
ALL_VIDEO_IDS = list(range(1, 81))

CACHE_IMG_SIZE = 256  # margine minimo sopra IMG_SIZE=224, per eventuali crop futuri
IMG_SIZE = 224
BATCH_SIZE = 64
EPOCHS = 15
PATIENCE = 4

Device: cuda


In [3]:
# --- 2. CARICAMENTO DATI E PRE-CACHING ---

def load_annotations(phase_dir, video_ids):
    annotations = {}
    missing = 0
    for vid in video_ids:
        pf = os.path.join(phase_dir, f"video{vid:02d}-phase.txt")
        if not os.path.exists(pf): 
            missing += 1
            continue
        df = pd.read_csv(pf, sep=r'\t|,|\s+', engine='python')
        frame_idx = df.iloc[:, 0].values.astype(int)
        phases = df.iloc[:, 1].map(PHASE_DICT).values.astype(int)
        
        # Sottocampionamento a 1 fps
        if len(frame_idx) > 1 and (frame_idx[1] - frame_idx[0]) == 1:
            mask = np.arange(0, len(frame_idx), 25)
            frame_idx = frame_idx[mask]
            phases = phases[mask]
            
        assert len(frame_idx) == len(set(frame_idx)), f"Indici frame duplicati nel video {vid}"
        annotations[vid] = {'frame_indices': frame_idx, 'phases': phases}
        
    print(f"Video caricati: {len(annotations)}/{len(video_ids)} (Mancanti: {missing})")
    return annotations

print("Caricamento annotazioni per training/validation CNN...")
train_anns = load_annotations(PHASE_DIR, CNN_TRAIN_IDS)
val_anns = load_annotations(PHASE_DIR, CNN_VAL_IDS)

print("Caricamento annotazioni per tutti gli 80 video (serve a cache + estrazione finale)...")
all_anns = load_annotations(PHASE_DIR, ALL_VIDEO_IDS)


def precompute_frame_cache(annotations, video_dir, cache_dir, img_size=CACHE_IMG_SIZE):
    """Estrae i frame in sequenza, li ridimensiona e li salva come JPEG.
    Usa un marker per-video scritto SOLO a fine estrazione completata,
    per evitare che un crash a metà video venga scambiato per 'già fatto'."""
    print("\nPre-caching dei frame come immagini JPEG (ridimensionate)...")
    os.makedirs(cache_dir, exist_ok=True)

    for vid, ann in tqdm(annotations.items(), desc="Estrazione Frame"):
        marker = os.path.join(cache_dir, f".done_v{vid:02d}")
        if os.path.exists(marker):
            continue  # video completato in un run precedente

        vp = os.path.join(video_dir, f"video{vid:02d}.mp4")
        if not os.path.exists(vp):
            continue

        # Controllo spazio disco PRIMA di iniziare questo video
        free_gb = shutil.disk_usage(cache_dir).free / 1e9
        if free_gb < 1.0:  # meno di 1GB libero: fermati con un errore chiaro
            raise RuntimeError(
                f"Spazio disco quasi esaurito ({free_gb:.2f} GB liberi) "
                f"prima di processare video {vid}. Interrompo per evitare corruzione silenziosa."
            )

        cap = cv2.VideoCapture(vp)
        target_set = set(ann['frame_indices'].tolist())
        max_frame = int(max(target_set))
        current_frame = 0

        while current_frame <= max_frame:
            if current_frame in target_set:
                ret, frame = cap.read()
                if ret:
                    frame_resized = cv2.resize(frame, (img_size, img_size), interpolation=cv2.INTER_AREA)
                    out_path = os.path.join(cache_dir, f"v{vid:02d}_f{current_frame:07d}.jpg")
                    cv2.imwrite(out_path, frame_resized, [int(cv2.IMWRITE_JPEG_QUALITY), 90])
            else:
                cap.grab()
            current_frame += 1
        cap.release()

        open(marker, 'w').close()  # marca il video come COMPLETATO solo ora

CACHE_INPUT_DIR = "/kaggle/input/datasets/christiancorvino/frame-cache-all80/frame_cache" # dataset pubblicato dopo la 1° run
if not os.path.exists(CACHE_INPUT_DIR):
    CACHE_INPUT_DIR = "/kaggle/input/frame-cache-all80/frame_cache"

if os.path.exists(CACHE_INPUT_DIR) and len(os.listdir(CACHE_INPUT_DIR)) > 0:
    print("Cache trovata come dataset Kaggle già pubblicato, salto la decodifica video.")
    CACHE_DIR = CACHE_INPUT_DIR
else:
    print("Cache non trovata come input, la costruisco ora (una tantum, copre tutti gli 80 video)...")
    precompute_frame_cache(all_anns, VIDEO_DIR, CACHE_DIR)  # <-- all_anns, non {**train_anns, **val_anns}
    
    # Report spazio occupato/disponibile dopo il caching
    used_gb = sum(os.path.getsize(os.path.join(CACHE_DIR, f)) for f in os.listdir(CACHE_DIR)) / 1e9
    free_gb = shutil.disk_usage(CACHE_DIR).free / 1e9
    print(f"Cache: {used_gb:.2f} GB usati, {free_gb:.2f} GB ancora liberi")


class Cholec80FrameDatasetCached(Dataset):
    """Dataset che legge i file JPEG estratti, garantendo I/O ultra-veloce"""
    def __init__(self, annotations, cache_dir, transform=None):
        self.samples = []
        self.transform = transform
        missing_frames = 0
        for vid, ann in annotations.items():
            for frame_idx, phase in zip(ann['frame_indices'], ann['phases']):
                path = os.path.join(cache_dir, f"v{vid:02d}_f{frame_idx:07d}.jpg")
                if os.path.exists(path):
                    self.samples.append({'path': path, 'phase': phase})
                else:
                    missing_frames += 1
        if missing_frames > 0:
            print(f"[Warning] {missing_frames} frame non trovati nella cache.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        # Carica JPEG
        frame = cv2.cvtColor(cv2.imread(s['path']), cv2.COLOR_BGR2RGB)
        img = Image.fromarray(frame)
        if self.transform:
            img = self.transform(img)
        return img, s['phase']

Caricamento annotazioni per training/validation CNN...
Video caricati: 27/27 (Mancanti: 0)
Video caricati: 5/5 (Mancanti: 0)
Caricamento annotazioni per tutti gli 80 video (serve a cache + estrazione finale)...
Video caricati: 80/80 (Mancanti: 0)
Cache trovata come dataset Kaggle già pubblicato, salto la decodifica video.


In [4]:
# --- 3. AUGMENTATION E SAMPLER ---

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = Cholec80FrameDatasetCached(train_anns, CACHE_DIR, transform=train_transform)
val_dataset = Cholec80FrameDatasetCached(val_anns, CACHE_DIR, transform=val_transform)

print(f"\nFrame di Training disponibili: {len(train_dataset)}")
print(f"Frame di Validation disponibili: {len(val_dataset)}")

# WeightedRandomSampler per sbilanciamento classi
train_labels = [s['phase'] for s in train_dataset.samples]
class_counts = np.bincount(train_labels, minlength=NUM_CLASSES)
class_weights = 1. / np.maximum(class_counts, 1)
sample_weights = [class_weights[label] for label in train_labels]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(train_dataset),
    replacement=True
)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, sampler=sampler,
    num_workers=0  # da 4 a 0: elimina il problema, il dataset ora legge JPEG cache (già veloce)
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=0
)


Frame di Training disponibili: 58492
Frame di Validation disponibili: 12540


In [5]:
# --- 4. MODELLO E FIX BATCHNORM ---

model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
num_ftrs = model.fc.in_features
model.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(num_ftrs, NUM_CLASSES))

for name, param in model.named_parameters():
    if 'layer3' in name or 'layer4' in name or 'fc' in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

model = model.to(DEVICE)

optimizer = optim.AdamW([
    {'params': model.layer3.parameters(), 'lr': 1e-4},
    {'params': model.layer4.parameters(), 'lr': 1e-4},
    {'params': model.fc.parameters(), 'lr': 1e-3}
], weight_decay=1e-4)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

TRAINABLE_PREFIXES = ('layer3', 'layer4', 'fc')  # deve combaciare col blocco requires_grad sopra

def freeze_bn_stats(model, trainable_prefixes):
    for name, module in model.named_modules():
        if isinstance(module, nn.BatchNorm2d) and not any(p in name for p in trainable_prefixes):
            module.eval()

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 197MB/s]


In [6]:
# --- 5. LOOP DI TRAINING FRAME-LEVEL ---

best_val_f1 = 0.0
best_model_state = None
patience_counter = 0

print("\nInizio Fine-Tuning della ResNet50...")
print(f"{'Ep':>4} {'TrLoss':>8} {'VlLoss':>8} {'VlAcc':>7} {'VlF1':>7}")
print("-" * 40)

for epoch in range(1, EPOCHS + 1):
    model.train()
    # FIX: model.train() sovrascrive tutto, dobbiamo rimettere in eval le BatchNorm congelate
    freeze_bn_stats(model, TRAINABLE_PREFIXES)
    
    tr_loss = 0.0
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch}", leave=False):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        tr_loss += loss.item() * images.size(0)
    tr_loss /= len(train_dataset)
    
    model.eval()
    vl_loss, vl_correct = 0.0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            vl_loss += loss.item() * images.size(0)
            preds = outputs.argmax(1)
            vl_correct += (preds == labels).sum().item()
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    vl_loss /= len(val_dataset)
    vl_acc = vl_correct / len(val_dataset)
    vl_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    
    star = ""
    if vl_f1 > best_val_f1:
        best_val_f1 = vl_f1
        best_model_state = deepcopy(model.state_dict())
        patience_counter = 0
        star = " *"
        per_class_f1 = f1_score(all_labels, all_preds, average=None, zero_division=0)
        for i, pname in enumerate(PHASE_DICT.keys()):
            print(f"      {pname:>25}: F1={per_class_f1[i]:.3f}")
    else:
        patience_counter += 1
    
    print(f"{epoch:4d} {tr_loss:8.4f} {vl_loss:8.4f} {vl_acc:7.4f} {vl_f1:7.4f}{star}")
    
    if patience_counter >= PATIENCE:
        print("Early stopping!")
        break

print(f"\nBest Validation F1: {best_val_f1:.4f}")
model.load_state_dict(best_model_state)

# Per assicurarti che i DataLoader vengano distrutti nel momento e nel processo giusti, 
# invece di essere lasciati alla garbage collection tardiva a fine notebook
del train_loader, val_loader
import gc
gc.collect()


Inizio Fine-Tuning della ResNet50...
  Ep   TrLoss   VlLoss   VlAcc    VlF1
----------------------------------------


Epoch 1:   0%|          | 0/914 [00:00<?, ?it/s]

                    Preparation: F1=0.688
        CalotTriangleDissection: F1=0.816
                ClippingCutting: F1=0.648
          GallbladderDissection: F1=0.687
           GallbladderPackaging: F1=0.657
            CleaningCoagulation: F1=0.642
          GallbladderRetraction: F1=0.510
   1   0.7129   1.0800  0.7251  0.6638 *


Epoch 2:   0%|          | 0/914 [00:00<?, ?it/s]

                    Preparation: F1=0.749
        CalotTriangleDissection: F1=0.816
                ClippingCutting: F1=0.643
          GallbladderDissection: F1=0.704
           GallbladderPackaging: F1=0.630
            CleaningCoagulation: F1=0.646
          GallbladderRetraction: F1=0.526
   2   0.5390   1.0826  0.7297  0.6733 *


...

Epoch 10:   0%|          | 0/914 [00:00<?, ?it/s]

  10   0.4704   1.0889  0.7523  0.6551


Epoch 11:   0%|          | 0/914 [00:00<?, ?it/s]

  11   0.4699   1.0816  0.7569  0.6504
Early stopping!

Best Validation F1: 0.6830


148

In [7]:
# --- 6. RI-ESTRAZIONE DELLE FEATURE PER TUTTI GLI 80 VIDEO ---

print("\n" + "="*50)
print("ESTRAZIONE FEATURE FINE-TUNED PER TUTTI GLI 80 VIDEO")
print("="*50)

model.fc = nn.Identity()
model.eval()

def extract_features_from_cache(model, annotations, cache_dir, output_dir, batch_size=64):
    os.makedirs(output_dir, exist_ok=True)
    for vid, ann in tqdm(annotations.items(), desc="Estraendo feature finali"):
        imgs, valid_phases = [], []
        for fi, phase in zip(ann['frame_indices'], ann['phases']):
            path = os.path.join(cache_dir, f"v{vid:02d}_f{fi:07d}.jpg")
            if os.path.exists(path):
                frame = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
                imgs.append(val_transform(Image.fromarray(frame)))
                valid_phases.append(phase)

        feats_all = []
        with torch.no_grad():
            for start in range(0, len(imgs), batch_size):
                batch = torch.stack(imgs[start:start+batch_size]).to(DEVICE)
                feats_all.append(model(batch).cpu().numpy())

        if feats_all:
            np.save(os.path.join(output_dir, f"video{vid:02d}_features.npy"), np.concatenate(feats_all))
            np.save(os.path.join(output_dir, f"video{vid:02d}_labels.npy"), np.array(valid_phases))

extract_features_from_cache(model, all_anns, CACHE_DIR, OUTPUT_DIR)
print("\nProcesso completato! Le feature sono in:", OUTPUT_DIR)


ESTRAZIONE FEATURE FINE-TUNED PER TUTTI GLI 80 VIDEO


Estraendo feature finali:   0%|          | 0/80 [00:00<?, ?it/s]


Processo completato! Le feature sono in: /kaggle/working/features_finetuned
